<a href="https://colab.research.google.com/github/24jr1a0507ui/AI_Sales_-_Inventory_dashboard/blob/main/AI_Sales_%26_Inventory_dashboard.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
from google.colab import files
uploaded=files.upload()  # A button appears—click it and select your CSV from your computer

Saving retail_sales.csv to retail_sales.csv


In [5]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [6]:
import pandas as pd
# Load your uploaded CSV file
df=pd.read_csv('retail_sales.csv')
# View the first few rows of your data
df.head()

,date,store_id,item_id,sales,price,promo,weekday,month
0,2019-01-01,store_1,item_1,41,21.30,0,1,1
1,2019-01-02,store_1,item_1,53,21.30,0,2,1
2,2019-01-03,store_1,item_1,39,21.30,0,3,1
3,2019-01-04,store_1,item_1,35,21.30,0,4,1
4,2019-01-05,store_1,item_1,51,17.04,1,5,1


In [7]:
# Check data types, row count, and missing values
print(df.info())
# Get a quick statistical summary of the sales and prices
print(df.describe())

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 4565000 entries, 0 to 4564999
Data columns (total 8 columns):
 #   Column    Dtype  
---  ------    -----  
 0   date      object 
 1   store_id  object 
 2   item_id   object 
 3   sales     int64  
 4   price     float64
 5   promo     int64  
 6   weekday   int64  
 7   month     int64  
dtypes: float64(1), int64(4), object(3)
memory usage: 278.6+ MB
None
              sales         price         promo       weekday         month
count  4.565000e+06  4.565000e+06  4.565000e+06  4.565000e+06  4.565000e+06
mean   2.926466e+01  5.399323e+01  9.999869e-02  3.001643e+00  6.523549e+00
std    1.500996e+01  2.578461e+01  2.999983e-01  1.999315e+00  3.448534e+00
min    0.000000e+00  8.020000e+00  0.000000e+00  0.000000e+00  1.000000e+00
25%    1.800000e+01  3.197000e+01  0.000000e+00  1.000000e+00  4.000000e+00
50%    2.700000e+01  5.352000e+01  0.000000e+00  3.000000e+00  7.000000e+00
75%    3.800000e+01  7.536000e+01  0.000000e+00  5.000000

In [8]:
import pandas as pd
import numpy as np
# 1. Convert date column to datetime type
df['date']=pd.to_datetime(df['date'])
# 2. Sort data chronologically to ensure proper time-series forecasting
df=df.sort_values(by=['store_id','item_id','date']).reset_index(drop=True)
# 3. Create historical "Lag" features (What were the sales 1 day ago and 7 days ago?)
# This helps the model see recent momentum.
df['lag_1']=df.groupby(['store_id','item_id'])['sales'].shift(1)
df['lag_7']=df.groupby(['store_id','item_id'])['sales'].shift(7)
# 4. Drop rows with missing values created by the lags
df.dropna(inplace=True)
print("Feature engineering complete! Shape of data:",df.shape)
df.head()

Feature engineering complete! Shape of data: (4547500, 10)


,date,store_id,item_id,sales,price,promo,weekday,month,lag_1,lag_7
7,2019-01-08,store_1,item_1,48,21.3,0,1,1,45.0,41.0
8,2019-01-09,store_1,item_1,50,21.3,0,2,1,48.0,53.0
9,2019-01-10,store_1,item_1,44,21.3,0,3,1,50.0,39.0
10,2019-01-11,store_1,item_1,38,21.3,0,4,1,44.0,35.0
11,2019-01-12,store_1,item_1,30,21.3,0,5,1,38.0,51.0


In [9]:
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, r2_score
# 1. Select Features (X) and Target Variable (y)
features=['price','promo','weekday','month','lag_1','lag_7']
X=df[features]
y=df['sales']
# 2. Take a sample to prevent Colab from crashing due to memory limits
X_sample,_,y_sample,_=train_test_split(X,y,train_size=0.05,random_state=42)
# 3. Split into training and testing sets
X_train,X_test,y_train,y_test=train_test_split(X_sample,y_sample,test_size=0.2,random_state=42)
# 4. Initialize and train the Model
print("Training the AI Sales Forecaster (this may take a minute)...")
model=RandomForestRegressor(n_estimators=50,max_depth=10,random_state=42,n_jobs=-1)
model.fit(X_train,y_train)
# 5. Evaluate the model
predictions=model.predict(X_test)
mae=mean_absolute_error(y_test,predictions)
r2=r2_score(y_test,predictions)
print(f"\nModel Training Complete!")
print(f"Mean Absolute Error (MAE): {mae:.2f} units")
print(f"R² Score (Accuracy Metric): {r2:.2f}")

Training the AI Sales Forecaster (this may take a minute)...

Model Training Complete!
Mean Absolute Error (MAE): 3.29 units
R² Score (Accuracy Metric): 0.92


In [10]:
# 1. Generate predictions for our entire dataset
df['predicted_sales']=model.predict(X)

# 2. Calculate standard deviation of sales per item to understand demand volatility
stock_summary=df.groupby(['store_id','item_id']).agg(
    avg_daily_prediction=('predicted_sales','mean'),
    sales_volatility=('sales','std')
).reset_index()

# 3. Assume a standard Supplier Lead Time of 3 days
lead_time_days=3

# 4. Calculate Safety Stock & Reorder Point
# Safety Stock = 1.65 (95% service level) * Volatility * sqrt(Lead Time)
stock_summary['safety_stock']=np.ceil(1.65*stock_summary['sales_volatility']*np.sqrt(lead_time_days))

# Reorder Point = (Average Daily Sales * Lead Time) + Safety Stock
stock_summary['reorder_point']=np.ceil((stock_summary['avg_daily_prediction']*lead_time_days)+stock_summary['safety_stock'])

print("--- Smart Inventory Recommendations (First 10 Items) ---")
print(stock_summary[['store_id','item_id','avg_daily_prediction','safety_stock','reorder_point']].head(10))

# Save the final results to a CSV file you can download for Power BI / Streamlit
stock_summary.to_csv('inventory_recommendations.csv',index=False)
print("\nSuccess! Saved dashboard data to 'inventory_recommendations.csv'")

--- Smart Inventory Recommendations (First 10 Items) ---
  store_id  item_id  avg_daily_prediction  safety_stock  reorder_point
0  store_1   item_1             44.169657          40.0          173.0
1  store_1  item_10             22.725960          22.0           91.0
2  store_1  item_11             24.638366          23.0           97.0
3  store_1  item_12             20.867186          20.0           83.0
4  store_1  item_13             39.710100          35.0          155.0
5  store_1  item_14             23.525131          22.0           93.0
6  store_1  item_15             21.292576          21.0           85.0
7  store_1  item_16             30.294314          28.0          119.0
8  store_1  item_17             16.556095          16.0           66.0
9  store_1  item_18             38.220819          34.0          149.0

Success! Saved dashboard data to 'inventory_recommendations.csv'


In [12]:
!pip install -q streamlit
!npm install -q -g localtunnel

⠙⠹⠸⠼⠴⠦⠧
changed 22 packages in 874ms
⠧
⠧3 packages are looking for funding
⠧  run `npm fund` for details
⠧

In [14]:
%%writefile app.py
import streamlit as st
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

st.set_page_config(page_title="AI Sales & Inventory Dashboard",layout="wide")

st.title("📦 AI-Powered Smart Sales Forecasting & Inventory Optimization")
st.markdown("Predicting future sales demand and recommending optimal inventory levels to eliminate overstocking and shortages.")

# Load the recommendations generated in Step 3
@st.cache_data
def load_data():
    return pd.read_csv('inventory_recommendations.csv')

try:
    df_res=load_data()

    # --- TOP METRICS ROW ---
    st.subheader("📊 Key Performance Metrics")
    col1,col2, col3=st.columns(3)
    col1.metric("Total Unique Items Tracked",f"{df_res['item_id'].nunique():,}")
    col2.metric("Total Active Stores",f"{df_res['store_id'].nunique():,}")
    col3.metric("Avg Daily Predicted Demand (Per Item)",f"{df_res['avg_daily_prediction'].mean():.2f} units")

    st.markdown("---")

    # --- STORE & PRODUCT VIEW ---
    st.subheader("🔍 Stock Level Recommendations by Product")

    # Filters
    selected_store=st.selectbox("Select Store ID",sorted(df_res['store_id'].unique()))
    store_data=df_res[df_res['store_id']==selected_store]

    # Display Table
    st.dataframe(
        store_data[['item_id','avg_daily_prediction','safety_stock','reorder_point']]
        .rename(columns={
            'item_id':'Item ID',
            'avg_daily_prediction':'AI Predicted Daily Sales',
            'safety_stock':'Required Safety Stock',
            'reorder_point':'Reorder Point (Trigger)'
        }),
        use_container_width=True
    )

    # --- VISUALIZATION ---
    st.subheader("📈 Visualizing Inventory Thresholds")
    top_10_items=store_data.head(10)
    fig,ax=plt.subplots(figsize=(10,4))
    x=np.arange(len(top_10_items['item_id']))
    width=0.35
    ax.bar(x-width/2,top_10_items['safety_stock'],width,label='Safety Stock',color='#FF6B6B')
    ax.bar(x+width/2,top_10_items['reorder_point'],width,label='Reorder Point',color='#4D96FF')
    ax.set_ylabel('Units')
    ax.set_title(f'Inventory Strategy: First 10 Items in {selected_store}')
    ax.set_xticks(x)
    ax.set_xticklabels(top_10_items['item_id'],rotation=45)
    ax.legend()
    st.pyplot(fig)
except FileNotFoundError:
    st.error("Could not find 'inventory_recommendations.csv'. Please make sure you successfully ran the model code step!")

Overwriting app.py


In [ ]:
# 1. Get your public IP address (needed for the security page)
print("\n--- INSTRUCTIONS TO OPEN DASHBOARD ---")
print("1. Click the 'url: https://...' link generated by localtunnel below.")
print("2. A webpage will open asking for an endpoint IP password.")
print("3. Copy and paste THIS exact IP address into that box and click submit:")
!curl ipv4.icanhazip.com

# 2. Start the Streamlit app and expose it via LocalTunnel
import subprocess
subprocess.Popen(["streamlit","run","app.py","--server.port","8501"])
!npx localtunnel --port 8501


--- INSTRUCTIONS TO OPEN DASHBOARD ---
1. Click the 'url: https://...' link generated by localtunnel below.
2. A webpage will open asking for an endpoint IP password.
3. Copy and paste THIS exact IP address into that box and click submit:
34.187.230.235
⠙⠹⠸⠼⠴⠦⠧⠇your url is: https://upset-things-matter.loca.lt
